# 01. ReAct Agent from Scratch

**Topics covered:** ReAct Pattern · Reasoning + Acting

This notebook starts the [**agents**](https://github.com/S33mi/modern-ai-llm-journey/blob/main/05_agents/) series.

We will:
1. Understand the **ReAct** loop (Reason + Act)
2. Define simple **tools** the agent can call
3. Build a minimal agent loop **from scratch** (no LangChain / LlamaIndex agent runtime)
4. Run multi-step examples with observation feedback
5. Keep it **CPU/GPU compatible**

## 1. What is ReAct?

ReAct (Yao et al., 2022) interleaves:

- **Thought** – the model reasons about what to do next
- **Action** – it calls a tool with arguments
- **Observation** – the environment returns a result

This repeats until the model produces a final answer.

```text
Thought: I need the capital of France.
Action: search[capital of France]
Observation: Paris
Thought: I have the answer.
Action: finish[Paris]
```

The pattern is powerful because the model can **ground** itself in tool outputs instead of hallucinating facts.

## 2. Setup

```bash
pip install transformers accelerate torch
```

In [22]:
#pip install transformers accelerate torch

In [23]:
import re
import json
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Small instruction-tuned model that works on CPU
MODEL_NAME = "google/flan-t5-base" if DEVICE == "cuda" else "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()
print(f"Model: {MODEL_NAME}")

Device: cpu


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model: google/flan-t5-small


## 3. Define Tools

Each tool is a Python function plus a short description the LLM can read.

In [24]:
def tool_calculator(expression: str) -> str:
    """Evaluate a simple math expression safely."""
    allowed = set("0123456789+-*/(). %")
    if not all(c in allowed for c in expression):
        return "Error: invalid characters in expression"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"


def tool_unit_convert(query: str) -> str:
    """
    Very small unit converter.
    Formats: '10 km to miles', '100 c to f', '5 kg to lb'
    """
    q = query.lower().strip()
    m = re.match(r"([0-9.]+)\s*([a-z]+)\s+to\s+([a-z]+)", q)
    if not m:
        return "Error: use format like '10 km to miles'"
    value, src, dst = float(m.group(1)), m.group(2), m.group(3)
    table = {
        ("km", "miles"): value * 0.621371,
        ("miles", "km"): value * 1.60934,
        ("kg", "lb"): value * 2.20462,
        ("lb", "kg"): value * 0.453592,
        ("c", "f"): value * 9 / 5 + 32,
        ("f", "c"): (value - 32) * 5 / 9,
    }
    key = (src, dst)
    if key not in table:
        return f"Error: unsupported conversion {src} → {dst}"
    return f"{table[key]:.4g} {dst}"


# Tiny in-memory "knowledge base" for a search tool
KB = {
    "capital of france": "Paris",
    "capital of japan": "Tokyo",
    "capital of germany": "Berlin",
    "inventor of the telephone": "Alexander Graham Bell",
    "speed of light": "approximately 299,792 km/s",
    "boiling point of water": "100°C at standard pressure",
}

def tool_search(query: str) -> str:
    """Search a tiny factual knowledge base."""
    q = query.lower().strip()
    for k, v in KB.items():
        if k in q or q in k:
            return v
    return "No result found in knowledge base."


TOOLS = {
    "calculator": {
        "fn": tool_calculator,
        "description": "Evaluate a math expression. Input example: 2+2*5",
    },
    "convert": {
        "fn": tool_unit_convert,
        "description": "Convert units. Input example: 10 km to miles",
    },
    "search": {
        "fn": tool_search,
        "description": "Search facts. Input example: capital of France",
    },
}

def tools_description() -> str:
    lines = []
    for name, meta in TOOLS.items():
        lines.append(f"- {name}: {meta['description']}")
    return "\n".join(lines)

print(tools_description())

- calculator: Evaluate a math expression. Input example: 2+2*5
- convert: Convert units. Input example: 10 km to miles
- search: Search facts. Input example: capital of France


## 4. Prompt Template for ReAct

We force a strict format so we can parse `Action` lines reliably:

```text
Thought: ...
Action: tool_name[argument]
```

or

```text
Thought: ...
Action: finish[final answer]
```

In [25]:
SYSTEM_PROMPT = f"""You are a ReAct agent. You solve tasks using tools.

Available tools:
{tools_description()}

Respond using EXACTLY this format every time:
Thought: <your reasoning>
Action: <tool_name>[<argument>]

When you know the final answer, use:
Thought: <reasoning>
Action: finish[<final answer>]

Do not add extra text outside this format.
"""

print(SYSTEM_PROMPT)

You are a ReAct agent. You solve tasks using tools.

Available tools:
- calculator: Evaluate a math expression. Input example: 2+2*5
- convert: Convert units. Input example: 10 km to miles
- search: Search facts. Input example: capital of France

Respond using EXACTLY this format every time:
Thought: <your reasoning>
Action: <tool_name>[<argument>]

When you know the final answer, use:
Thought: <reasoning>
Action: finish[<final answer>]

Do not add extra text outside this format.



## 5. LLM Call + Action Parser

In [26]:
def llm(prompt: str, max_new_tokens: int = 128) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()


ACTION_RE = re.compile(
    r"Action\s*:\s*([a-zA-Z_]+)\s*\[\s*(.*?)\s*\]",
    re.IGNORECASE | re.DOTALL,
)

def parse_action(text: str):
    """Return (tool_name, argument) or (None, None)."""
    m = ACTION_RE.search(text)
    if not m:
        return None, None
    return m.group(1).lower().strip(), m.group(2).strip()


# Sanity check
sample = "Thought: I should calculate.\nAction: calculator[3*(4+5)]"
print(parse_action(sample))

('calculator', '3*(4+5)')


## 6. The ReAct Loop (from scratch)

In [27]:
def react_agent(question: str, max_steps: int = 10, verbose: bool = True) -> str:
    """
    Minimal ReAct agent.
    Returns the final answer string (or last observation on failure).
    """
    history = f"Question: {question}\n"

    for step in range(1, max_steps + 1):
        prompt = SYSTEM_PROMPT + "\n" + history + "\nThought:"
        # Model continues after "Thought:"
        raw = llm(prompt)
        # Ensure we have a Thought/Action style block
        if not raw.lower().startswith("thought"):
            block = "Thought: " + raw
        else:
            block = raw

        if verbose:
            print(f"--- Step {step} ---")
            print(block)

        tool_name, arg = parse_action(block)

        if tool_name is None:
            # Try to salvage a finish-style free answer
            history += block + "\n"
            if verbose:
                print("Observation: (could not parse action)")
            continue

        if tool_name == "finish":
            if verbose:
                print(f"Final answer: {arg}")
            return arg

        if tool_name not in TOOLS:
            observation = f"Error: unknown tool '{tool_name}'"
        else:
            observation = TOOLS[tool_name]["fn"](arg)

        if verbose:
            print(f"Observation: {observation}\n")

        history += block + f"\nObservation: {observation}\n"

    return "Max steps reached without finish[]."

print("ReAct loop defined.")

ReAct loop defined.


## 7. Run Examples

In [28]:
print("=== Math ===")
ans = react_agent("What is 17 * 23?")
print("→", ans)

=== Math ===
--- Step 1 ---
Thought: 17 * 23
Observation: (could not parse action)
--- Step 2 ---
Thought: a calculator
Observation: (could not parse action)
--- Step 3 ---
Thought: calculator
Observation: (could not parse action)
--- Step 4 ---
Thought: calculator
Observation: (could not parse action)
--- Step 5 ---
Thought: calculator
Observation: (could not parse action)
--- Step 6 ---
Thought: calculator
Observation: (could not parse action)
--- Step 7 ---
Thought: calculator
Observation: (could not parse action)
--- Step 8 ---
Thought: calculator
Observation: (could not parse action)
--- Step 9 ---
Thought: calculator
Observation: (could not parse action)
--- Step 10 ---
Thought: calculator
Observation: (could not parse action)
→ Max steps reached without finish[].


In [29]:
print("=== Unit conversion ===")
ans = react_agent("Convert 10 km to miles")
print("→", ans)

=== Unit conversion ===
--- Step 1 ---
Thought: 10 km to miles
Observation: (could not parse action)
--- Step 2 ---
Thought: 10 km to miles
Observation: (could not parse action)
--- Step 3 ---
Thought: 10 km to miles
Observation: (could not parse action)
--- Step 4 ---
Thought: 10 km to miles
Observation: (could not parse action)
--- Step 5 ---
Thought: 10 km to miles
Observation: (could not parse action)
--- Step 6 ---
Thought: 10 km to miles
Observation: (could not parse action)
--- Step 7 ---
Thought: 10 km to miles
Observation: (could not parse action)
--- Step 8 ---
Thought: 10 km to miles
Observation: (could not parse action)
--- Step 9 ---
Thought: 10 km to miles
Observation: (could not parse action)
--- Step 10 ---
Thought: 10 km to miles
Observation: (could not parse action)
→ Max steps reached without finish[].


In [30]:
print("=== Fact search ===")
ans = react_agent("What is the capital of Japan?")
print("→", ans)

=== Fact search ===
--- Step 1 ---
Thought: 
Observation: (could not parse action)
--- Step 2 ---
Thought: Japan
Observation: (could not parse action)
--- Step 3 ---
Thought: Japan
Observation: (could not parse action)
--- Step 4 ---
Thought: France
Observation: (could not parse action)
--- Step 5 ---
Thought: France
Observation: (could not parse action)
--- Step 6 ---
Thought: France
Observation: (could not parse action)
--- Step 7 ---
Thought: Japan
Observation: (could not parse action)
--- Step 8 ---
Thought: Japan
Observation: (could not parse action)
--- Step 9 ---
Thought: Japan
Observation: (could not parse action)
--- Step 10 ---
Thought: Japan
Observation: (could not parse action)
→ Max steps reached without finish[].


In [31]:
print("=== Multi-step (search then maybe finish) ===")
ans = react_agent("Who invented the telephone?")
print("→", ans)

=== Multi-step (search then maybe finish) ===
--- Step 1 ---
Thought: a reAct agent
Observation: (could not parse action)
--- Step 2 ---
Thought: a reAct agent
Observation: (could not parse action)
--- Step 3 ---
Thought: a reAct agent
Observation: (could not parse action)
--- Step 4 ---
Thought: a reAct agent
Observation: (could not parse action)
--- Step 5 ---
Thought: a reAct agent
Observation: (could not parse action)
--- Step 6 ---
Thought: reAct agent
Observation: (could not parse action)
--- Step 7 ---
Thought: reAct agent
Observation: (could not parse action)
--- Step 8 ---
Thought: reAct agent
Observation: (could not parse action)
--- Step 9 ---
Thought: reAct agent
Observation: (could not parse action)
--- Step 10 ---
Thought: reAct agent
Observation: (could not parse action)
→ Max steps reached without finish[].


## 8. Why Small Models Struggle (and how to improve)

FLAN-T5-small/base is enough to **demonstrate** the loop, but it often:

- drifts from the strict `Action: tool[arg]` format
- calls the wrong tool
- finishes too early

Improved FLAN-T5-small/base tool version on this notebook is here: [01_react_agent_from_scratch_flan.ipynb](https://github.com/S33mi/modern-ai-llm-journey/blob/main/05_agents/01_react_agent_from_scratch_flan.ipynb)

**Ways to improve in production:**

1. Use a stronger instruction / chat model (Llama 3, Mistral, Qwen, GPT-class APIs)
2. Few-shot examples of valid Thought/Action traces in the prompt
3. Constrained decoding / JSON tool calls (OpenAI-style function calling)
4. Separate **planner** model and **executor**
5. Add memory (conversation buffer, vector memory)

The next notebook covers **tool calling** in a more structured way.

## 9. Optional: Few-Shot ReAct Prompt

Adding one worked example often stabilizes tiny models.

In [32]:
FEWSHOT = """
Example:
Question: What is 5 + 7?
Thought: I should use the calculator.
Action: calculator[5+7]
Observation: 12
Thought: I have the result.
Action: finish[12]
"""

def react_agent_fewshot(question: str, max_steps: int = 5, verbose: bool = True) -> str:
    history = FEWSHOT + f"\nQuestion: {question}\n"
    for step in range(1, max_steps + 1):
        prompt = SYSTEM_PROMPT + "\n" + history + "\nThought:"
        raw = llm(prompt)
        block = raw if raw.lower().startswith("thought") else "Thought: " + raw
        if verbose:
            print(f"--- Step {step} ---")
            print(block)
        tool_name, arg = parse_action(block)
        if tool_name is None:
            history += block + "\n"
            continue
        if tool_name == "finish":
            if verbose:
                print(f"Final answer: {arg}")
            return arg
        if tool_name not in TOOLS:
            observation = f"Error: unknown tool '{tool_name}'"
        else:
            observation = TOOLS[tool_name]["fn"](arg)
        if verbose:
            print(f"Observation: {observation}\n")
        history += block + f"\nObservation: {observation}\n"
    return "Max steps reached without finish[]."


print("=== Few-shot math ===")
print("→", react_agent_fewshot("What is 9 * 8?"))

=== Few-shot math ===
--- Step 1 ---
Thought: 9 * 8?
--- Step 2 ---
Thought: a calculator
--- Step 3 ---
Thought: Action: calculator
--- Step 4 ---
Thought: calculator
--- Step 5 ---
Thought: calculator
→ Max steps reached without finish[].


## 10. Summary

| Piece | Role |
|-------|------|
| **Thought** | Natural-language reasoning step |
| **Action** | Tool name + argument (or `finish[...]`) |
| **Observation** | Tool result fed back into the prompt |
| **Loop** | Repeat until `finish` or max steps |
| **Tools** | Plain Python functions + descriptions |

### Minimal ReAct skeleton

```python
history = f"Question: {question}\n"
for step in range(max_steps):
    text = llm(SYSTEM + history + "\nThought:")
    tool, arg = parse_action(text)
    if tool == "finish":
        return arg
    obs = TOOLS[tool](arg)
    history += text + f"\nObservation: {obs}\n"
```

---

Improved FLAN-T5-small/base tool version on this notebook is here: [01_react_agent_from_scratch_flan.ipynb](https://github.com/S33mi/modern-ai-llm-journey/blob/main/05_agents/01_react_agent_from_scratch_flan.ipynb)

---

**Next notebook:** [`02_tool_calling_agent.ipynb`](https://github.com/S33mi/modern-ai-llm-journey/blob/main/05_agents/02_tool_calling_agent.ipynb.ipynb)
Tool Calling · Function Calling · Memory

---
**For contribution and insihght:** [**S33mi**](https://github.com/S33mi)

Open to Data Science/Analytics and ML/AI related opportunities